# ANM Mode-Drive Pipeline Test

**Input:** Initial PDB ID + Target PDB ID (e.g., 1AKE + 4AKE)

**What this notebook does:**
1. Fetches PDB structures (initial & target)
2. Runs ANM Mode-Drive pipeline from initial
3. Compares trajectory against target via TM-score & RMSD
4. Compares GNM from PDB coords vs GNM from z_ij (converter)
5. Plots: B-factors, contact maps, RMSD/TM trajectory, df evolution, collectivity
6. Downloads all figures as ZIP

## 1. Environment Setup (sadece bir kere çalıştır)

Git clone, pip install, OF3 model download. Runtime yeniden başlamadıkça tekrar çalıştırmaya gerek yok.

In [ ]:
# ════════════════════════════════════════════════════
#  ONE-TIME SETUP — run once per runtime
# ════════════════════════════════════════════════════
import os, sys, shutil

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Always fresh clone ANM-openfold3
if os.path.exists('/content/ANM-openfold3'):
    shutil.rmtree('/content/ANM-openfold3')
!git clone https://github.com/beratkaskaloglu/ANM-openfold3.git /content/ANM-openfold3

# Clone and install OpenFold3
if not os.path.exists('/content/ANM-openfold3/openfold3-repo'):
    !git clone https://github.com/aqlaboratory/openfold-3.git /content/ANM-openfold3/openfold3-repo
    !pip install -e /content/ANM-openfold3/openfold3-repo -q
else:
    try:
        import openfold3
    except ImportError:
        !pip install -e /content/ANM-openfold3/openfold3-repo -q

!pip install -q biopython matplotlib numpy torch

sys.path.insert(0, '/content/ANM-openfold3')
sys.path.insert(0, '/content/ANM-openfold3/openfold3-repo')

# Ensure OF3 config dir exists
os.makedirs(os.path.expanduser('~/.openfold3'), exist_ok=True)

# Pre-download OF3 model parameters
from pathlib import Path
from openfold3.entry_points.parameters import (
    download_model_parameters,
    get_default_checkpoint_dir,
    DEFAULT_CHECKPOINT_NAME,
)
_param_dir = get_default_checkpoint_dir()
_param_dir.mkdir(parents=True, exist_ok=True)
download_model_parameters(_param_dir, DEFAULT_CHECKPOINT_NAME, skip_confirmation=True)
print(f"OF3 model parameters ready at {_param_dir}")

# Verify
import importlib
importlib.invalidate_caches()
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src') or mod_name.startswith('openfold3'):
        del sys.modules[mod_name]
sys.path.insert(0, '/content/ANM-openfold3')
sys.path.insert(0, '/content/ANM-openfold3/openfold3-repo')

print("Environment setup complete.")

## 2. Config (her denemede değiştir)

Parametreleri değiştirip buradan aşağısını tekrar çalıştır.

In [ ]:
# ════════════════════════════════════════════════════
#  CONFIGURABLE PARAMETERS
# ════════════════════════════════════════════════════

# ── PDB Structure ──────────────────────────────────
INITIAL_PDB = "1AKE"          # Open conformation
TARGET_PDB  = "4AKE"          # Closed conformation
CHAIN_ID    = "A"             # Chain to extract

# ── Mode-Drive Pipeline ───────────────────────────
N_STEPS          = 30           # Fixed number of steps
N_ANM_MODES      = 20
MAX_COMBO_SIZE   = 3           # Max modes per combination
COMBINATION_STRATEGY = "manual" # "collectivity", "manual", "targeted", "grid", "random"
MANUAL_MODES     = (0, 1, 2)   # Mode indices for manual strategy
DF               = 0.8         # Global displacement factor (A)
DF_MIN           = 0.3
DF_MAX           = 4.0
DF_ESCALATION    = 1.5
Z_MIXING_ALPHA   = 0.3
Z_DIRECTION      = "plus"      # "plus" = +delta_z, "minus" = -delta_z
N_COMBINATIONS   = 20
ANM_CUTOFF       = 15.0        # ANM Hessian contact cutoff (A)
CONTACT_R_CUT    = 10.0        # Sigmoid contact cutoff (A)
CONTACT_TAU      = 1.5         # Sigmoid temperature

# ── OF3 Diffusion ─────────────────────────────────
USE_MSA_SERVER   = True         # ColabFold MSA server
USE_TEMPLATES    = False        # Template search
NUM_ROLLOUT_STEPS = None        # None = default (200 full rollout)

# ── Paths ──────────────────────────────────────────
DRIVE_DIR        = "/content/drive/MyDrive/ANM-openfold3/checkpoints/finetune_msa"
CHECKPOINT_SELECTION = "best_bf_r"  # "best_bf_r" or "best_val_loss"
FIGURE_DIR       = "/content/figures"

## 3. Checkpoint Selection & Model Load

Checkpoint seçimi, converter yükleme. Config'e bağlı — config değişince buradan itibaren tekrar çalıştır.

In [ ]:
# ════════════════════════════════════════════════════
#  CHECKPOINT SELECTION + CONVERTER + PDB FETCH
# ════════════════════════════════════════════════════
import warnings
warnings.filterwarnings('ignore')

import os, json, importlib
import numpy as np
import torch

# Force reload src modules
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

from src.converter import PairContactConverter
from src.mode_drive import ModeDrivePipeline, ModeDriveConfig, compute_rmsd, tm_score, make_pseudo_diffusion

# ── Checkpoint selection ───────────────────────────
history_path = os.path.join(DRIVE_DIR, 'history.json')
best_model_path = os.path.join(DRIVE_DIR, 'best_model.pt')

if CHECKPOINT_SELECTION == 'best_bf_r' and os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    
    best_bf_r = -1
    best_epoch = -1
    for entry in history:
        bf_r = entry.get('val_bf_pearson', 0)
        if bf_r > best_bf_r:
            best_bf_r = bf_r
            best_epoch = entry['epoch']
    
    print(f'Best bf_r: {best_bf_r:.4f} at epoch {best_epoch}')
    
    epoch_ckpt = os.path.join(DRIVE_DIR, f'epoch_{best_epoch:04d}.pt')
    
    if os.path.exists(epoch_ckpt):
        CHECKPOINT_PATH = epoch_ckpt
        print(f'Using epoch checkpoint: {epoch_ckpt}')
    else:
        ckpt = torch.load(best_model_path, map_location='cpu', weights_only=False)
        ckpt_bf = ckpt.get('val_bf_pearson', 'N/A')
        ckpt_epoch = ckpt.get('epoch', 'N/A')
        print(f'best_model.pt: epoch={ckpt_epoch}, bf_r={ckpt_bf}')
        
        available = sorted([f for f in os.listdir(DRIVE_DIR) if f.startswith('epoch_') and f.endswith('.pt')])
        print(f'Available epoch checkpoints: {available}')
        
        best_available = None
        best_available_bf = -1
        for fname in available:
            fpath = os.path.join(DRIVE_DIR, fname)
            ck = torch.load(fpath, map_location='cpu', weights_only=False)
            ck_bf = ck.get('val_bf_pearson', 0)
            ck_epoch = ck.get('epoch', '?')
            print(f'  {fname}: epoch={ck_epoch}, bf_r={ck_bf:.4f}')
            if ck_bf > best_available_bf:
                best_available_bf = ck_bf
                best_available = fpath
        
        if best_available and best_available_bf > float(ckpt_bf or 0):
            CHECKPOINT_PATH = best_available
            print(f'\nSelected: {best_available} (bf_r={best_available_bf:.4f})')
        else:
            CHECKPOINT_PATH = best_model_path
            print(f'\nUsing best_model.pt (bf_r={ckpt_bf})')
else:
    CHECKPOINT_PATH = best_model_path
    print(f'Using best_model.pt')

# Show checkpoint info
ckpt_info = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
print(f'\n{"═"*50}')
print(f'Selected checkpoint: {os.path.basename(CHECKPOINT_PATH)}')
print(f'  Epoch:       {ckpt_info.get("epoch", "?")}')
print(f'  Val loss:    {ckpt_info.get("val_loss", "?")}')
print(f'  Val bf_r:    {ckpt_info.get("val_bf_pearson", "?")}')
print(f'  Val adj_acc: {ckpt_info.get("val_adj_acc", "?")}')
print(f'{"═"*50}')

# ── Load converter ─────────────────────────────────
converter = PairContactConverter(CHECKPOINT_PATH, device='cpu')
print(f"\nConverter loaded from {CHECKPOINT_PATH}")

# ── Fetch PDB structures ──────────────────────────
from Bio.PDB import PDBParser, PDBList

def fetch_ca_coords(pdb_id: str, chain_id: str = "A") -> tuple:
    """Download PDB and extract CA coordinates + B-factors."""
    pdbl = PDBList()
    fname = pdbl.retrieve_pdb_file(pdb_id, pdir='/tmp/pdb', file_format='pdb')
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, fname)
    chain = structure[0][chain_id]
    
    three_to_one = {
        'ALA':'A','CYS':'C','ASP':'D','GLU':'E','PHE':'F',
        'GLY':'G','HIS':'H','ILE':'I','LYS':'K','LEU':'L',
        'MET':'M','ASN':'N','PRO':'P','GLN':'Q','ARG':'R',
        'SER':'S','THR':'T','VAL':'V','TRP':'W','TYR':'Y'
    }
    cas, bfs, seq = [], [], []
    for residue in chain:
        if residue.id[0] != ' ' or 'CA' not in residue:
            continue
        ca = residue['CA']
        cas.append(ca.get_vector().get_array())
        bfs.append(ca.get_bfactor())
        seq.append(three_to_one.get(residue.get_resname(), 'X'))
    
    return (torch.tensor(np.array(cas), dtype=torch.float32),
            torch.tensor(bfs, dtype=torch.float32),
            ''.join(seq))

print(f"\nFetching {INITIAL_PDB} (initial)...")
initial_coords, initial_bfactors, initial_seq = fetch_ca_coords(INITIAL_PDB, CHAIN_ID)
print(f"  → {initial_coords.shape[0]} residues")

print(f"Fetching {TARGET_PDB} (target)...")
target_coords, target_bfactors, target_seq = fetch_ca_coords(TARGET_PDB, CHAIN_ID)
print(f"  → {target_coords.shape[0]} residues")

N = min(initial_coords.shape[0], target_coords.shape[0])
initial_coords = initial_coords[:N]
target_coords = target_coords[:N]
initial_bfactors = initial_bfactors[:N]
target_bfactors = target_bfactors[:N]
print(f"\nUsing N={N} residues")
print(f"Sequence match: {initial_seq[:N] == target_seq[:N]}")

## 4. ANM & GNM Analysis

In [4]:
from src.anm import build_hessian, anm_modes, anm_bfactors, collectivity
from src.kirchhoff import soft_kirchhoff, gnm_decompose
from src.coords_to_contact import coords_to_contact

def full_analysis(coords, label=""):
    """Run ANM + GNM analysis on a set of CA coordinates."""
    print(f"\n{'='*50}")
    print(f"  Analysis: {label}")
    print(f"{'='*50}")
    
    # ANM
    H = build_hessian(coords, cutoff=ANM_CUTOFF)
    anm_evals, anm_evecs = anm_modes(H, N_ANM_MODES)
    anm_bf = anm_bfactors(anm_evals, anm_evecs)
    anm_coll = collectivity(anm_evecs)
    
    print(f"  ANM: {anm_evals.shape[0]} modes")
    print(f"  ANM eigenvalues (first 5): {anm_evals[:5].tolist()}")
    print(f"  ANM collectivity (first 5): {anm_coll[:5].tolist()}")
    
    # Contact map
    contact = coords_to_contact(coords, CONTACT_R_CUT, CONTACT_TAU)
    
    # GNM
    gamma = soft_kirchhoff(contact)
    gnm_evals, gnm_evecs, gnm_bf = gnm_decompose(gamma, N_ANM_MODES)
    
    print(f"  GNM: {gnm_evals.shape[0]} modes")
    
    # B-factor correlation
    anm_bf_n = (anm_bf - anm_bf.mean()) / (anm_bf.std() + 1e-8)
    gnm_bf_n = (gnm_bf - gnm_bf.mean()) / (gnm_bf.std() + 1e-8)
    anm_gnm_r = (anm_bf_n * gnm_bf_n).mean().item()
    print(f"  ANM-GNM B-factor Pearson r: {anm_gnm_r:.3f}")
    
    return {
        'anm_evals': anm_evals, 'anm_evecs': anm_evecs,
        'anm_bf': anm_bf, 'anm_coll': anm_coll,
        'gnm_evals': gnm_evals, 'gnm_evecs': gnm_evecs, 'gnm_bf': gnm_bf,
        'contact': contact,
    }

initial_analysis = full_analysis(initial_coords, f"{INITIAL_PDB} (initial)")
target_analysis = full_analysis(target_coords, f"{TARGET_PDB} (target)")


  Analysis: 1AKE (initial)
  ANM: 20 modes
  ANM eigenvalues (first 5): [1.094520926475525, 1.3352015018463135, 1.7695505619049072, 1.9290308952331543, 2.206662893295288]
  ANM collectivity (first 5): [0.3957505524158478, 0.3116697072982788, 0.13805802166461945, 0.09516336768865585, 0.16355597972869873]
  GNM: 20 modes
  ANM-GNM B-factor Pearson r: 0.709

  Analysis: 4AKE (target)
  ANM: 20 modes
  ANM eigenvalues (first 5): [0.04419810324907303, 0.09797948598861694, 0.2092682123184204, 0.3381592035293579, 0.5369920134544373]
  ANM collectivity (first 5): [0.43288081884384155, 0.40923696756362915, 0.4012228548526764, 0.5499144196510315, 0.3838466703891754]
  GNM: 20 modes
  ANM-GNM B-factor Pearson r: 0.838


## 5. Roundtrip & GNM Comparison

In [ ]:
# Test roundtrip: real contact → z_pseudo → contact_recon
contact_init = initial_analysis['contact']
z_from_contact = converter.contact_to_z(contact_init)
recon_contact, z_recon, roundtrip_mse = converter.roundtrip(z_from_contact)
print(f"Roundtrip MSE (contact→z→contact→z): {roundtrip_mse:.6f}")
print(f"Contact recon MSE: {((contact_init - recon_contact) ** 2).mean():.6f}")

In [ ]:
# GNM comparison: direct contact vs converter roundtrip contact
# Direct: coords → contact → GNM
# Converter: coords → contact → inverse(contact) → z_pseudo → forward(z_pseudo) → contact_recon → GNM
# This tests whether the bottleneck (W_enc, v, W_dec) preserves GNM spectral info

direct_contact = initial_analysis['contact']

# Roundtrip through converter's learned weights
z_from_contact = converter.contact_to_z(direct_contact)           # inverse: C → z_pseudo
contact_recon = converter.z_to_contact(z_from_contact)            # forward: z_pseudo → C_recon

print(f"Contact roundtrip MSE: {((direct_contact - contact_recon) ** 2).mean():.6f}")
print(f"Contact recon range: [{contact_recon.min():.3f}, {contact_recon.max():.3f}]")

# GNM on direct contact
direct_gnm = converter.analyze(direct_contact, n_modes=N_ANM_MODES)

# GNM on roundtrip contact (tests bottleneck fidelity)
recon_gnm = converter.analyze(contact_recon, n_modes=N_ANM_MODES)

# Compare B-factors
direct_bf = direct_gnm['b_factors']
recon_bf = recon_gnm['b_factors']

direct_n = (direct_bf - direct_bf.mean()) / (direct_bf.std() + 1e-8)
recon_n = (recon_bf - recon_bf.mean()) / (recon_bf.std() + 1e-8)
roundtrip_r = (direct_n * recon_n).mean().item()
print(f"\nDirect GNM vs Roundtrip GNM B-factor Pearson r: {roundtrip_r:.3f}")
print("(This measures how well the bottleneck preserves GNM spectral information)")

## 6. OF3 Diffusion & Pipeline Run

In [ ]:
# ════════════════════════════════════════════════════
#  OF3 Diffusion Setup
# ════════════════════════════════════════════════════
import json as _json

_query_dir = "/content/of3_queries"
os.makedirs(_query_dir, exist_ok=True)

# Get sequence from PDB
from Bio.PDB import PDBParser as _Parser, PDBList as _PDBList
from Bio.SeqUtils import seq1 as _seq1

_parser = _Parser(QUIET=True)
_pdbl = _PDBList()
_pdb_file = _pdbl.retrieve_pdb_file(INITIAL_PDB, pdir="/tmp/pdb_cache", file_format="pdb")
_structure = _parser.get_structure(INITIAL_PDB, _pdb_file)
_chain = [c for c in _structure[0].get_chains() if c.id == CHAIN_ID][0]
_sequence = "".join(
    _seq1(res.get_resname()) for res in _chain
    if res.get_id()[0] == " " and "CA" in res
)

_query = {
    "queries": {
        INITIAL_PDB: {
            "chains": [{
                "molecule_type": "protein",
                "chain_ids": [CHAIN_ID],
                "sequence": _sequence,
            }]
        }
    }
}
_query_path = os.path.join(_query_dir, f"{INITIAL_PDB}.json")
with open(_query_path, 'w') as f:
    _json.dump(_query, f, indent=2)

print(f"OF3 query written: {_query_path} (seq len={len(_sequence)})")

# Load OF3 diffusion function + trunk zij
from src.of3_diffusion import load_of3_diffusion

diffusion_fn, zij_trunk = load_of3_diffusion(
    query_json=_query_path,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    use_msa_server=USE_MSA_SERVER,
    use_templates=USE_TEMPLATES,
    num_rollout_steps=NUM_ROLLOUT_STEPS,
)

print(f"\nzij_trunk shape: {zij_trunk.shape}")
print(f"zij_trunk stats: mean={zij_trunk.mean():.4f}, std={zij_trunk.std():.4f}")

# ════════════════════════════════════════════════════
#  GNM Comparison: trunk zij vs PDB coords
# ════════════════════════════════════════════════════
# 1. PDB coords → contact (direct) → GNM
direct_contact = initial_analysis['contact']
direct_gnm = converter.analyze(direct_contact, n_modes=N_ANM_MODES)

# 2. Trunk zij → forward head → contact → GNM
trunk_contact = converter.z_to_contact(zij_trunk)
trunk_gnm = converter.analyze(trunk_contact, n_modes=N_ANM_MODES)

# Compare B-factors
direct_bf = direct_gnm['b_factors']
trunk_bf = trunk_gnm['b_factors']

direct_n = (direct_bf - direct_bf.mean()) / (direct_bf.std() + 1e-8)
trunk_n = (trunk_bf - trunk_bf.mean()) / (trunk_bf.std() + 1e-8)
trunk_r = (direct_n * trunk_n).mean().item()

# Compare contacts
contact_mse = ((direct_contact - trunk_contact) ** 2).mean().item()

print(f"\n{'═'*60}")
print(f"  OF3 Trunk z_ij vs PDB Direct Comparison")
print(f"{'═'*60}")
print(f"  Contact MSE (trunk vs PDB):          {contact_mse:.6f}")
print(f"  GNM B-factor Pearson r:              {trunk_r:.3f}")
print(f"  Trunk contact range:                 [{trunk_contact.min():.3f}, {trunk_contact.max():.3f}]")
print(f"  Direct contact range:                [{direct_contact.min():.3f}, {direct_contact.max():.3f}]")
print(f"{'═'*60}")

# ════════════════════════════════════════════════════
#  Configure Pipeline with REAL OF3 diffusion
# ════════════════════════════════════════════════════
config = ModeDriveConfig(
    n_anm_modes=N_ANM_MODES,
    n_steps=N_STEPS,
    combination_strategy=COMBINATION_STRATEGY,
    manual_modes=MANUAL_MODES,
    n_combinations=N_COMBINATIONS,
    z_mixing_alpha=Z_MIXING_ALPHA,
    z_direction=Z_DIRECTION,
    df=DF,
    df_min=DF_MIN,
    df_max=DF_MAX,
    df_escalation_factor=DF_ESCALATION,
    max_combo_size=MAX_COMBO_SIZE,
    anm_cutoff=ANM_CUTOFF,
    contact_r_cut=CONTACT_R_CUT,
    contact_tau=CONTACT_TAU,
)

pipeline = ModeDrivePipeline(converter, config, diffusion_fn=diffusion_fn)
print(f"\nPipeline configured WITH OF3 diffusion")
print(f"  z_mod → OF3 SampleDiffusion → all-atom → CA coords")
print(f"  strategy={COMBINATION_STRATEGY}, z_direction={Z_DIRECTION}")
if COMBINATION_STRATEGY == "manual":
    print(f"  manual_modes={MANUAL_MODES}")
print(f"  n_steps={N_STEPS}, df_min={DF_MIN}, df_max={DF_MAX}")
print(f"  max_combo_size={MAX_COMBO_SIZE}")

In [ ]:
# Run pipeline (verbose=True prints live RMSD + TM-score per step)
print("Running Mode-Drive pipeline...\n")
result = pipeline.run(initial_coords, zij_trunk, target_coords=target_coords, verbose=True)

print(f"\nTotal steps: {result.total_steps}")
print(f"Trajectory: {len(result.trajectory)} structures")

## 7. TM-score Calculation

In [ ]:
def tm_score(coords_model: torch.Tensor, coords_ref: torch.Tensor) -> float:
    """Approximate TM-score between two CA coordinate sets.
    
    TM-score = (1/N) * Σ_i 1 / (1 + (d_i / d0)^2)
    where d0 = 1.24 * (N-15)^(1/3) - 1.8
    
    Note: This is a simplified version without optimal superposition.
    For publication-quality results, use TMalign.
    """
    N = coords_model.shape[0]
    d0 = 1.24 * (N - 15) ** (1.0/3.0) - 1.8
    d0 = max(d0, 0.5)  # floor
    
    # Per-residue distances
    d = (coords_model - coords_ref).norm(dim=-1)  # [N]
    
    # TM-score
    tm = (1.0 / (1.0 + (d / d0) ** 2)).mean().item()
    return tm

# Compute TM-scores for trajectory
print("TM-scores (vs target):")
tm_scores_target = []
tm_scores_initial = []
rmsd_from_initial = [0.0]  # initial → initial = 0
rmsd_from_target = [compute_rmsd(target_coords, initial_coords)]

tm_init = tm_score(initial_coords, target_coords)
tm_scores_target.append(tm_init)
tm_scores_initial.append(1.0)  # self
print(f"  Initial: TM={tm_init:.3f}  RMSD_target={rmsd_from_target[0]:.2f}Å")

for i, step in enumerate(result.step_results):
    tm_t = tm_score(step.new_ca, target_coords)
    tm_i = tm_score(step.new_ca, initial_coords)
    rmsd_t = compute_rmsd(target_coords, step.new_ca)
    
    tm_scores_target.append(tm_t)
    tm_scores_initial.append(tm_i)
    rmsd_from_initial.append(step.rmsd)
    rmsd_from_target.append(rmsd_t)
    
    print(f"  Step {i+1}: TM_target={tm_t:.3f}  TM_initial={tm_i:.3f}  "
          f"RMSD_initial={step.rmsd:.2f}Å  RMSD_target={rmsd_t:.2f}Å")

## 8. Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({
    'figure.figsize': (14, 10),
    'font.size': 11,
    'axes.titlesize': 13,
})

def save_fig(fig, name):
    path = os.path.join(FIGURE_DIR, f"{name}.png")
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f"  Saved: {path}")

In [ ]:
# ════════════════════════════════════════════════════
#  FIGURE 1: RMSD & TM-score Trajectory
# ════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
steps = list(range(len(rmsd_from_initial)))
labels = ['Initial'] + [f'Step {i+1}' for i in range(len(result.step_results))]

# RMSD plot
ax = axes[0]
ax.plot(steps, rmsd_from_initial, 'o-', color='#2196F3', label='RMSD from initial', linewidth=2)
ax.plot(steps, rmsd_from_target, 's-', color='#F44336', label='RMSD from target', linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('RMSD (Å)')
ax.set_title(f'RMSD Trajectory: {INITIAL_PDB} → explore')
ax.legend()
ax.set_xticks(steps)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.grid(alpha=0.3)

# TM-score plot
ax = axes[1]
ax.plot(steps, tm_scores_target, 'o-', color='#F44336', label=f'TM vs {TARGET_PDB}', linewidth=2)
ax.plot(steps, tm_scores_initial, 's-', color='#2196F3', label=f'TM vs {INITIAL_PDB}', linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('TM-score')
ax.set_title('TM-score Trajectory')
ax.legend()
ax.set_xticks(steps)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)

fig.tight_layout()
save_fig(fig, '01_rmsd_tm_trajectory')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════
#  FIGURE 2: df Evolution & Collectivity per Step
# ════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

step_idx = list(range(1, len(result.step_results) + 1))
dfs_used = [s.df_used for s in result.step_results]
colls = [s.combo.collectivity_score for s in result.step_results]
combo_labels = [s.combo.label for s in result.step_results]

# df evolution
ax = axes[0]
bars = ax.bar(step_idx, dfs_used, color='#FF9800', alpha=0.8, edgecolor='#E65100')
ax.axhline(y=DF_MIN, color='green', linestyle='--', alpha=0.5, label=f'df_min={DF_MIN}')
ax.axhline(y=DF_MAX, color='red', linestyle='--', alpha=0.5, label=f'df_max={DF_MAX}')
ax.set_xlabel('Step')
ax.set_ylabel('df used (Å)')
ax.set_title('Displacement Factor Evolution')
ax.legend()
ax.grid(alpha=0.3, axis='y')

# Collectivity per step
ax = axes[1]
bars = ax.bar(step_idx, colls, color='#9C27B0', alpha=0.8, edgecolor='#6A1B9A')
for i, (bar, label) in enumerate(zip(bars, combo_labels)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            label.split('_m')[-1] if '_m' in label else label,
            ha='center', va='bottom', fontsize=8, rotation=30)
ax.set_xlabel('Step')
ax.set_ylabel('Collectivity (κ)')
ax.set_title('Selected Combo Collectivity')
ax.set_ylim(0, 1.0)
ax.grid(alpha=0.3, axis='y')

fig.tight_layout()
save_fig(fig, '02_df_collectivity')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════
#  FIGURE 3: B-factor Comparison (4-way)
#  Initial PDB | Target PDB | ANM initial | GNM initial | Converter GNM
# ════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
residues = np.arange(1, N + 1)

# Normalize all B-factors to [0, 1] for comparison
def norm01(x):
    x = x.detach().cpu().numpy()
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

# Panel 1: Initial structure B-factors
ax = axes[0]
ax.plot(residues, norm01(initial_bfactors), '-', color='black', label=f'{INITIAL_PDB} PDB B-factor', linewidth=1.5)
ax.plot(residues, norm01(initial_analysis['anm_bf']), '-', color='#2196F3', label='ANM B-factor', linewidth=1.2, alpha=0.8)
ax.plot(residues, norm01(initial_analysis['gnm_bf']), '-', color='#4CAF50', label='GNM B-factor (from coords)', linewidth=1.2, alpha=0.8)
ax.plot(residues, norm01(converter_gnm['b_factors']), '--', color='#FF9800', label='GNM B-factor (from converter z)', linewidth=1.2, alpha=0.8)
ax.set_title(f'{INITIAL_PDB} B-factor Comparison')
ax.set_xlabel('Residue')
ax.set_ylabel('Normalized B-factor')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 2: Target structure B-factors
ax = axes[1]
ax.plot(residues, norm01(target_bfactors), '-', color='black', label=f'{TARGET_PDB} PDB B-factor', linewidth=1.5)
ax.plot(residues, norm01(target_analysis['anm_bf']), '-', color='#F44336', label='ANM B-factor', linewidth=1.2, alpha=0.8)
ax.plot(residues, norm01(target_analysis['gnm_bf']), '-', color='#9C27B0', label='GNM B-factor (from coords)', linewidth=1.2, alpha=0.8)
ax.set_title(f'{TARGET_PDB} B-factor Comparison')
ax.set_xlabel('Residue')
ax.set_ylabel('Normalized B-factor')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

fig.tight_layout()
save_fig(fig, '03_bfactor_comparison')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════
#  FIGURE 4: Contact Map Comparison
#  Initial | Target | Difference | Last step
# ════════════════════════════════════════════════════

last_step = result.step_results[-1]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

maps = [
    (initial_analysis['contact'], f'{INITIAL_PDB} Contact'),
    (target_analysis['contact'], f'{TARGET_PDB} Contact'),
    ((target_analysis['contact'] - initial_analysis['contact']).abs(), 'Difference |T - I|'),
    (last_step.contact_map, f'Step {N_STEPS} Contact'),
]

for ax, (cmap, title) in zip(axes, maps):
    data = cmap.detach().cpu().numpy()
    im = ax.imshow(data, cmap='viridis', vmin=0, vmax=1, aspect='equal')
    ax.set_title(title)
    ax.set_xlabel('Residue')
    ax.set_ylabel('Residue')
    plt.colorbar(im, ax=ax, fraction=0.046)

fig.tight_layout()
save_fig(fig, '04_contact_maps')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════
#  FIGURE 5: Per-mode Collectivity Spectrum
# ════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Initial
ax = axes[0]
coll_init = initial_analysis['anm_coll'].detach().cpu().numpy()
modes = np.arange(1, len(coll_init) + 1)
ax.bar(modes, coll_init, color='#2196F3', alpha=0.8)
ax.set_xlabel('Mode Index (non-trivial)')
ax.set_ylabel('Collectivity (κ)')
ax.set_title(f'{INITIAL_PDB} Mode Collectivity')
ax.set_ylim(0, 1.0)
ax.grid(alpha=0.3, axis='y')

# Target
ax = axes[1]
coll_tgt = target_analysis['anm_coll'].detach().cpu().numpy()
modes_t = np.arange(1, len(coll_tgt) + 1)
ax.bar(modes_t, coll_tgt, color='#F44336', alpha=0.8)
ax.set_xlabel('Mode Index (non-trivial)')
ax.set_ylabel('Collectivity (κ)')
ax.set_title(f'{TARGET_PDB} Mode Collectivity')
ax.set_ylim(0, 1.0)
ax.grid(alpha=0.3, axis='y')

fig.tight_layout()
save_fig(fig, '05_collectivity_spectrum')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════
#  FIGURE 6: Per-step B-factor Evolution
# ════════════════════════════════════════════════════

n_panels = min(N_STEPS, 5)
fig, axes = plt.subplots(1, n_panels, figsize=(4 * n_panels, 4), sharey=True)
if n_panels == 1:
    axes = [axes]

for i, (ax, step) in enumerate(zip(axes, result.step_results[:n_panels])):
    bf = norm01(step.b_factors)
    ax.plot(residues, bf, '-', color=plt.cm.viridis(i / n_panels), linewidth=1.2)
    ax.plot(residues, norm01(initial_bfactors), '--', color='gray', alpha=0.4, linewidth=0.8)
    ax.set_title(f'Step {i+1}\nRMSD={step.rmsd:.1f}Å df={step.df_used:.2f}')
    ax.set_xlabel('Residue')
    if i == 0:
        ax.set_ylabel('Norm B-factor')
    ax.grid(alpha=0.3)

fig.suptitle('B-factor Evolution Through Trajectory', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '06_bfactor_evolution')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════
#  FIGURE 7: 2D Distance Difference Maps
#  How distance matrix changes through trajectory
# ════════════════════════════════════════════════════

def dist_matrix(coords):
    return torch.cdist(coords.unsqueeze(0), coords.unsqueeze(0)).squeeze(0)

d_initial = dist_matrix(initial_coords)
d_target = dist_matrix(target_coords)

n_panels = min(N_STEPS + 1, 4)
fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))

# Initial vs target
diff_init = (d_initial - d_target).detach().cpu().numpy()
vmax = max(abs(diff_init.min()), abs(diff_init.max()))

im = axes[0].imshow(diff_init, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='equal')
axes[0].set_title(f'{INITIAL_PDB} - {TARGET_PDB}\ndist difference')
plt.colorbar(im, ax=axes[0], fraction=0.046)

# Trajectory steps vs target
for i in range(min(N_STEPS, n_panels - 1)):
    step = result.step_results[i]
    d_step = dist_matrix(step.new_ca)
    diff = (d_step - d_target).detach().cpu().numpy()
    im = axes[i+1].imshow(diff, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='equal')
    axes[i+1].set_title(f'Step {i+1} - {TARGET_PDB}\nRMSD_t={rmsd_from_target[i+1]:.1f}Å')
    plt.colorbar(im, ax=axes[i+1], fraction=0.046)

for ax in axes:
    ax.set_xlabel('Residue')
    ax.set_ylabel('Residue')

fig.tight_layout()
save_fig(fig, '07_distance_difference')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════
#  FIGURE 8: Summary Dashboard
# ════════════════════════════════════════════════════

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# (0,0) RMSD trajectory
ax = fig.add_subplot(gs[0, 0])
ax.plot(steps, rmsd_from_initial, 'o-', color='#2196F3', label='From initial', linewidth=2)
ax.plot(steps, rmsd_from_target, 's-', color='#F44336', label='From target', linewidth=2)
ax.set_title('RMSD Trajectory')
ax.set_ylabel('RMSD (Å)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# (0,1) TM-score trajectory
ax = fig.add_subplot(gs[0, 1])
ax.plot(steps, tm_scores_target, 'o-', color='#F44336', label=f'vs {TARGET_PDB}', linewidth=2)
ax.set_title('TM-score vs Target')
ax.set_ylabel('TM-score')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# (0,2) df + collectivity
ax = fig.add_subplot(gs[0, 2])
ax.bar(step_idx, dfs_used, color='#FF9800', alpha=0.6, label='df')
ax2 = ax.twinx()
ax2.plot(step_idx, colls, 'D-', color='#9C27B0', label='κ', linewidth=2)
ax.set_title('df & Collectivity')
ax.set_ylabel('df (Å)', color='#FF9800')
ax2.set_ylabel('κ', color='#9C27B0')
ax2.set_ylim(0, 1.0)
ax.grid(alpha=0.3)

# (1,0) Initial B-factor
ax = fig.add_subplot(gs[1, 0])
ax.plot(residues, norm01(initial_bfactors), 'k-', label='PDB', linewidth=1.2)
ax.plot(residues, norm01(initial_analysis['anm_bf']), '-', color='#2196F3', label='ANM', alpha=0.7)
ax.plot(residues, norm01(initial_analysis['gnm_bf']), '-', color='#4CAF50', label='GNM', alpha=0.7)
ax.set_title(f'{INITIAL_PDB} B-factors')
ax.set_ylabel('Normalized')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

# (1,1) Contact map initial
ax = fig.add_subplot(gs[1, 1])
im = ax.imshow(initial_analysis['contact'].detach().cpu().numpy(), cmap='viridis', vmin=0, vmax=1)
ax.set_title(f'{INITIAL_PDB} Contact Map')
plt.colorbar(im, ax=ax, fraction=0.046)

# (1,2) Contact map last step
ax = fig.add_subplot(gs[1, 2])
im = ax.imshow(last_step.contact_map.detach().cpu().numpy(), cmap='viridis', vmin=0, vmax=1)
ax.set_title(f'Step {N_STEPS} Contact Map')
plt.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle(f'ANM Mode-Drive: {INITIAL_PDB} → Exploration ({N_STEPS} steps)', fontsize=15, y=1.01)
save_fig(fig, '08_summary_dashboard')
plt.show()

## 9. Summary Table

In [ ]:
# ════════════════════════════════════════════════════
#  Summary Table
# ════════════════════════════════════════════════════

print("\n" + "═" * 85)
print(f"  ANM Mode-Drive Pipeline Summary: {INITIAL_PDB} → {TARGET_PDB}")
print("═" * 85)
print(f"{'Step':>6} {'RMSD_init':>10} {'RMSD_tgt':>10} {'TM_tgt':>8} {'df':>6} {'κ':>6} {'Combo':>20}")
print("-" * 85)
print(f"{'Init':>6} {'0.00':>10} {rmsd_from_target[0]:>10.2f} {tm_scores_target[0]:>8.3f} {'—':>6} {'—':>6} {'—':>20}")

for i, step in enumerate(result.step_results):
    print(f"{i+1:>6} {step.rmsd:>10.2f} {rmsd_from_target[i+1]:>10.2f} "
          f"{tm_scores_target[i+1]:>8.3f} {step.df_used:>6.2f} "
          f"{step.combo.collectivity_score:>6.3f} {step.combo.label:>20}")

print("═" * 85)
print(f"\nConfig: n_steps={N_STEPS}, df_min={DF_MIN}, df_max={DF_MAX}, "
      f"max_combo_size={MAX_COMBO_SIZE}, alpha={Z_MIXING_ALPHA}")

## 10. Download Figures

In [ ]:
# ════════════════════════════════════════════════════
#  Pack all figures into ZIP and download
# ════════════════════════════════════════════════════

zip_path = f"/content/mode_drive_{INITIAL_PDB}_{TARGET_PDB}.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(FIGURE_DIR)):
        fpath = os.path.join(FIGURE_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added: {fname}")

print(f"\nZIP created: {zip_path}")

# Download
try:
    from google.colab import files
    files.download(zip_path)
    print("Download started!")
except ImportError:
    print("Not in Colab — ZIP file saved locally.")